In [ ]:
import numpy as np
from scipy.optimize import minimize
import optimize
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

In [3]:
# Little random example with 3 assets and cov marrix
mu = np.array([0.08, 0.10, 0.12])
cov = np.array([
    [0.18, 0.03, 0.04],
    [0.03, 0.12, 0.02],
    [0.04, 0.02, 0.10],
])

sr_solution = optimize.maximize_sharpe_ratio(mu, cov, risk_free=0.02)
gm_solution = optimize.maximize_geometric_mean(mu, cov)

np.set_printoptions(suppress=False, precision=6)

print("Sharpe weights:", np.round(sr_solution["weights"], 6))
print("Optimal Sharpe:", np.round(sr_solution["optimal_sharpe"], 6))
print("GM weights:", np.round(gm_solution["weights"], 6))
print("Optimal GM:", np.round(gm_solution["optimal_gm"], 6))


Sharpe weights: [0.03751  0.352587 0.609903]
Optimal Sharpe: 0.362629
GM weights: [0.       0.313963 0.686037]
Optimal GM: 0.083822


In [32]:
# Define your ETF ticker, start date, and end date
tickers = ["VWCE.DE", "IUSN.DE"]  
start_date = "2020-01-01"
end_date = "2025-12-31"

# Download the historical data
data = yf.download(tickers, start=start_date, end=end_date)
timeseries = data['Close']

[*********************100%***********************]  2 of 2 completed


In [33]:
returns = timeseries.pct_change().dropna()

daily_mu = returns.mean()
daily_cov = returns.cov()
annual_mu = daily_mu * 252
annual_cov = daily_cov * 252
annual_vol = returns.std() * np.sqrt(252)
annual_return = (1 + returns).prod() ** (252 / len(returns)) - 1

stats = pd.DataFrame(
    {
        "historical_return": annual_return,
        "arithmetic_return": annual_mu,
        "volatility": annual_vol,
    }
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

mu, cov = daily_mu.to_numpy(), daily_cov.to_numpy()

sharpe = optimize.maximize_sharpe_ratio(mu, cov)
gm = optimize.maximize_geometric_mean(mu, cov)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")
print("\nPortfolio weights:")
print(weights.round(4))
print("\nOptimal daily Sharpe:", round(sharpe["optimal_sharpe"], 6))
print("Approx. annualized Sharpe:", round(sharpe["optimal_sharpe"] * np.sqrt(252), 6))
print("Optimal daily GM:", round(gm["optimal_gm"], 6))
print("Approx. annualized GM:", round((1 + gm["optimal_gm"]) ** 252 - 1, 6))



Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
historical_return   0.0780   0.1098
arithmetic_return   0.0949   0.1181
volatility          0.1982   0.1662

Portfolio weights:
         Sharpe_weight  GM_weight
ETF                              
IUSN.DE            0.0        0.0
VWCE.DE            1.0        1.0

Optimal daily Sharpe: 0.044745
Approx. annualized Sharpe: 0.71031
Optimal daily GM: 0.000414
Approx. annualized GM: 0.109856
